# M1 MMD investigation (+ atlas-reference UMAP)

This notebook investigates the M1 OOD experiment (single-population holdout `CL:0000875`, Pearson HVG), with M2 alongside as a reference. Two parts:

- **§1–§4 — MMD anatomy:** how MMD depends on the RBF bandwidth (gamma), its kernel-term decomposition, the sample-size (`ncells`) interaction, and the MMD-floor-vs-ncells table.
- **§5 — atlas-reference UMAP:** the training-atlas UMAP with the OOD holdout and the scGen / IMPACT_CellOT predicted-human clouds projected onto it (ported from `presentation_preparation.ipynb`), so the M1 prediction layout can be inspected once training + eval finish.

---

## Gamma sensitivity of MMD — what each RBF bandwidth contributes

The eval (`cellot/cellot_gpu/scripts/evaluate.py`) reports MMD as the **average over 50 RBF bandwidths** `gammas = np.logspace(1, -3, num=50)` (10 → 0.001). That single scalar hides *where* in bandwidth-space the discrepancy lives. This notebook decomposes the averaged score into its per-gamma pieces.

**What we look at**
1. **MMD vs gamma** for IMPACT / scGen / floor on one axis — which bandwidths drive the score (large gamma saturates to ~0; small gamma captures global location; the action is usually mid-range).
2. **Decomposition** of `MMD = ⟨k(x,x)⟩ + ⟨k(y,y)⟩ − 2⟨k(x,y)⟩` per gamma — the mechanism behind the curve.
3. **MMD vs gamma at ncells = 30/50/80** — how sample size interacts with bandwidth (the finite-sample bias behind the "MMD floor").

**Data**: the exact `treated` (real human OOD) and `imputed` (predicted) clouds are pre-dumped to `eval_clouds.npz` next to each model's `evals.csv` (via `cellot/cellot_gpu/scripts/dump_eval_clouds.py`, run in the CellOT env). This notebook is pure numpy/matplotlib and re-generates the `.npz` on demand if missing.

> Scope note: this is exploratory only — per the 2026-06-02 decision we are **not** writing a per-gamma breakdown into `evals.csv`.

In [1]:
import os
import subprocess
from pathlib import Path

import numpy as np
import matplotlib.pyplot as plt
from sklearn.metrics.pairwise import euclidean_distances

REPO = Path("/n/holylabs/mooney_lab/Lab/junyizhou/speciesOT")
CELL_GPU = REPO / "cellot/cellot_gpu"
OUT_DIR = REPO / "speciesOT/baseline/analysis/m1_pearson_outputs"
OUT_DIR.mkdir(parents=True, exist_ok=True)

# Match the eval EXACTLY (cellot/cellot_gpu/scripts/evaluate.py:63) so the
# per-gamma curve averages back to the number in evals.csv.
GAMMAS = np.logspace(1, -3, num=50)
N_REPS = 20  # more reps than the eval's 10 -> smoother curves

# CellOT-env python; only invoked to regenerate a missing eval_clouds.npz.
CELLOT_PY = "/n/home01/jzhou1125/.conda/envs/CellOT/bin/python"
EVALPREFIX = "evals_ood_data_space"

# Models analyzable now. M1 IMPACT is added once its training finishes.
MODELS = {
    "M2 IMPACT": "results/hvg_pearson_residuals_m2_ood/impact_cellot",
    "M2 scGen":  "results/hvg_pearson_residuals_m2_ood/scgen",
    "M1 scGen":  "results/hvg_pearson_residuals_m1_ood/scgen",
    "M1 IMPACT": "results/hvg_pearson_residuals_m1_ood/impact_cellot",  # uncomment when trained
}


def ensure_clouds(model_subdir):
    """Load (treated, imputed); regenerate the .npz via the CellOT env if absent."""
    p = CELL_GPU / model_subdir / EVALPREFIX / "eval_clouds.npz"
    if not p.exists():
        print(f"regenerating {p} ...")
        subprocess.run(
            [CELLOT_PY, "scripts/dump_eval_clouds.py", "--outdir", f"./{model_subdir}",
             "--setting", "ood", "--where", "data_space", "--embedding", "ae",
             "--evalprefix", EVALPREFIX],
            cwd=str(CELL_GPU), env=dict(os.environ, PYTHONPATH=str(CELL_GPU)), check=True,
        )
    d = np.load(p)
    return d["treated"], d["imputed"]


# ---- MMD machinery (env-agnostic; mirrors cellot.losses.mmd.mmd_distance) ----

def mmd_terms_per_gamma(x, y, gammas):
    """Per-gamma (xx, yy, xy, mmd) for two clouds. Computes squared distances
    once, then exponentiates per gamma (fast)."""
    Dxx = euclidean_distances(x, x, squared=True)
    Dyy = euclidean_distances(y, y, squared=True)
    Dxy = euclidean_distances(x, y, squared=True)
    xx = np.array([np.exp(-g * Dxx).mean() for g in gammas])
    yy = np.array([np.exp(-g * Dyy).mean() for g in gammas])
    xy = np.array([np.exp(-g * Dxy).mean() for g in gammas])
    return xx, yy, xy, (xx + yy - 2 * xy)


def _draw_model(treated, imputed, ncells, rng):
    ia = rng.choice(len(treated), ncells, replace=False)
    ib = rng.choice(len(imputed), ncells, replace=False)
    return treated[ia], imputed[ib]


def _draw_floor(treated, ncells, rng):
    """Disjoint split-half draw from the real cells (the MMD floor)."""
    ia = rng.choice(len(treated), ncells, replace=False)
    rem = np.setdiff1d(np.arange(len(treated)), ia)
    ib = (rng.choice(rem, ncells, replace=False)
          if len(rem) >= ncells else rng.choice(len(treated), ncells, replace=False))
    return treated[ia], treated[ib]


def gamma_curve(treated, imputed, gammas, ncells, n_reps, kind="model", seed=0):
    """Average per-gamma MMD (and terms) over n_reps subsample draws.
    kind='model' -> treated vs imputed; kind='floor' -> treated vs treated."""
    rng = np.random.default_rng(seed)
    mmds, xxs, yys, xys = [], [], [], []
    for _ in range(n_reps):
        if kind == "model":
            x, y = _draw_model(treated, imputed, ncells, rng)
        else:
            x, y = _draw_floor(treated, ncells, rng)
        xx, yy, xy, mmd = mmd_terms_per_gamma(x, y, gammas)
        mmds.append(mmd); xxs.append(xx); yys.append(yy); xys.append(xy)
    return {
        "mmd": np.mean(mmds, 0), "mmd_std": np.std(mmds, 0),
        "xx": np.mean(xxs, 0), "yy": np.mean(yys, 0), "xy": np.mean(xys, 0),
    }


clouds = {name: ensure_clouds(sub) for name, sub in MODELS.items()}
for name, (t, i) in clouds.items():
    print(f"{name:10s}: treated {t.shape}  imputed {i.shape}")

M2 IMPACT : treated (248, 1000)  imputed (261, 1000)
M2 scGen  : treated (248, 1000)  imputed (261, 1000)
M1 scGen  : treated (207, 1000)  imputed (219, 1000)
M1 IMPACT : treated (207, 1000)  imputed (219, 1000)


## 1. MMD vs gamma — IMPACT vs scGen vs floor (M2 OOD)

For each bandwidth `gamma`, the MMD term is computed separately (no averaging) at `ncells=80`. The horizontal mean of each curve equals the scalar MMD in `evals.csv`. The floor curve uses split-halves of the *real* cells, so the gap between a model curve and the floor curve, at each gamma, is that bandwidth's contribution to the real error.

In [2]:
ncells = 80

t_imp, i_imp = clouds["M2 IMPACT"]
t_sg, i_sg = clouds["M2 scGen"]

imp = gamma_curve(t_imp, i_imp, GAMMAS, ncells, N_REPS, kind="model")
sg = gamma_curve(t_sg, i_sg, GAMMAS, ncells, N_REPS, kind="model")
flr = gamma_curve(t_imp, None, GAMMAS, ncells, N_REPS, kind="floor")  # treated identical across models

fig, ax = plt.subplots(figsize=(8, 5))
ax.plot(GAMMAS, imp["mmd"], "-o", ms=3, label="IMPACT (treated vs imputed)")
ax.plot(GAMMAS, sg["mmd"], "-s", ms=3, label="scGen (treated vs imputed)")
ax.plot(GAMMAS, flr["mmd"], "--", color="k", label="floor (treated vs treated)")
ax.set_xscale("log")
ax.set_xlabel("gamma (RBF bandwidth)")
ax.set_ylabel(f"per-gamma MMD term  (ncells={ncells})")
ax.set_title("M2 OOD: where in bandwidth-space the MMD lives")
ax.legend(); ax.grid(alpha=0.3)
fig.tight_layout(); fig.savefig(OUT_DIR / "gamma_curve_m2_ood.png", dpi=150)
plt.show()

print("avg-over-gamma MMD (ties back to evals.csv):")
print(f"  IMPACT {imp['mmd'].mean():.4f}   scGen {sg['mmd'].mean():.4f}   floor {flr['mmd'].mean():.4f}")
print(f"  peak-contribution gamma: IMPACT={GAMMAS[np.argmax(imp['mmd']-flr['mmd'])]:.4g}  "
      f"scGen={GAMMAS[np.argmax(sg['mmd']-flr['mmd'])]:.4g}")

avg-over-gamma MMD (ties back to evals.csv):
  IMPACT 0.1063   scGen 0.1439   floor 0.0240
  peak-contribution gamma: IMPACT=0.005429  scGen=0.006551


In [3]:
# M1 (single-population holdout) — scGen now; IMPACT overlays automatically once
# its row appears in `clouds` (uncomment it in the setup cell after training).
ncells = 80
fig, ax = plt.subplots(figsize=(8, 5))
for name in ["M1 scGen", "M1 IMPACT"]:
    if name not in clouds:
        continue
    t, i = clouds[name]
    c = gamma_curve(t, i, GAMMAS, ncells, N_REPS, kind="model")
    ax.plot(GAMMAS, c["mmd"], "-o", ms=3, label=f"{name}  (avg MMD={c['mmd'].mean():.3f})")
t_m1 = clouds["M1 scGen"][0]
flr_m1 = gamma_curve(t_m1, None, GAMMAS, ncells, N_REPS, kind="floor")
ax.plot(GAMMAS, flr_m1["mmd"], "--", color="k", label=f"M1 floor (avg={flr_m1['mmd'].mean():.3f})")
ax.set_xscale("log")
ax.set_xlabel("gamma (RBF bandwidth)"); ax.set_ylabel(f"per-gamma MMD term (ncells={ncells})")
ax.set_title("M1 OOD (single holdout CL:0000875)")
ax.legend(); ax.grid(alpha=0.3)
fig.tight_layout(); fig.savefig(OUT_DIR / "gamma_curve_m1_ood.png", dpi=150)
plt.show()

## 2. Decomposition: `MMD = ⟨k(x,x)⟩ + ⟨k(y,y)⟩ − 2⟨k(x,y)⟩`

The mechanism behind the curve. Each kernel mean is bounded in (0, 1]:
- At **large gamma** the kernel is so narrow that off-diagonal similarities vanish; all three means collapse toward the same tiny value (only the i=i diagonal of xx/yy survives), so MMD → small.
- At **small gamma** the kernel is so wide that every pair looks identical (~1); xx ≈ yy ≈ xy ≈ 1, so MMD → 0.
- In between, `xy` (cross-cloud similarity) drops below `xx`,`yy` (within-cloud) by exactly the amount the two clouds are separated — that bump is the signal.

In [4]:
ncells = 80
t_imp, i_imp = clouds["M2 IMPACT"]
d = gamma_curve(t_imp, i_imp, GAMMAS, ncells, N_REPS, kind="model")

fig, ax = plt.subplots(figsize=(8, 5))
ax.plot(GAMMAS, d["xx"], label="⟨k(x,x)⟩ within real")
ax.plot(GAMMAS, d["yy"], label="⟨k(y,y)⟩ within imputed")
ax.plot(GAMMAS, d["xy"], label="⟨k(x,y)⟩ cross")
ax.plot(GAMMAS, d["mmd"], "--k", lw=2, label="MMD = xx + yy − 2·xy")
ax.set_xscale("log")
ax.set_xlabel("gamma (RBF bandwidth)"); ax.set_ylabel("kernel mean")
ax.set_title("M2 IMPACT: kernel-term decomposition vs gamma")
ax.legend(); ax.grid(alpha=0.3)
fig.tight_layout(); fig.savefig(OUT_DIR / "gamma_decomposition_m2_impact.png", dpi=150)
plt.show()

## 3. How ncells interacts with gamma

Same M2 IMPACT model and floor, evaluated at `ncells = 30 / 50 / 80`. The model curves barely move with sample size, but the **floor** curves drop sharply as ncells grows — that's the finite-sample bias of the MMD estimator shrinking. The vertical gap (model − floor) at fixed gamma is the sample-size-robust error.

In [5]:
t_imp, i_imp = clouds["M2 IMPACT"]
fig, ax = plt.subplots(figsize=(8, 5))
colors = {30: "tab:blue", 50: "tab:orange", 80: "tab:green"}
for nc in [30, 50, 80]:
    m = gamma_curve(t_imp, i_imp, GAMMAS, nc, N_REPS, kind="model")
    f = gamma_curve(t_imp, None, GAMMAS, nc, N_REPS, kind="floor")
    ax.plot(GAMMAS, m["mmd"], "-", color=colors[nc], label=f"model ncells={nc}")
    ax.plot(GAMMAS, f["mmd"], "--", color=colors[nc], alpha=0.7, label=f"floor ncells={nc}")
ax.set_xscale("log")
ax.set_xlabel("gamma (RBF bandwidth)"); ax.set_ylabel("per-gamma MMD term")
ax.set_title("M2 IMPACT: MMD vs gamma at ncells 30/50/80 (solid=model, dashed=floor)")
ax.legend(ncol=2, fontsize=8); ax.grid(alpha=0.3)
fig.tight_layout(); fig.savefig(OUT_DIR / "gamma_curve_m2_impact_by_ncells.png", dpi=150)
plt.show()

## 4. The MMD floor vs ncells — and why the curve dips at *both* gamma extremes

`mmd = ⟨k(x,x)⟩ + ⟨k(y,y)⟩ − 2⟨k(x,y)⟩`, with `k(x,y) = exp(−γ·‖x−y‖²)`. MMD is a **difference** of similarities (within-cloud minus cross-cloud), so a stricter kernel does **not** imply a higher MMD:

- **γ → large (strict):** no two distinct cells are similar → all off-diagonal kernel values → 0. Only the i=i diagonal (k=1) survives, so `⟨k(x,x)⟩ → 1/n`, `⟨k(y,y)⟩ → 1/n`, `⟨k(x,y)⟩ → 0`, hence **MMD → 2/n** — a constant fixed by sample size, blind to cloud shape. At n=80, 2/80 ≈ 0.025, exactly where the curves flatten. The model "matching the floor" here is the metric going blind (both treated-vs-imputed and treated-vs-treated collapse to the same 2/n diagonal), **not** good performance.
- **γ → small (loose):** every pair looks identical (k≈1) → all three terms ≈1 → MMD → 0 (blind the other way).

The discriminative signal lives in the **middle** (γ≈0.005–0.01), where the bandwidth ≈ the real cloud separation — hence the 50-gamma average.

The table below is the averaged-over-gamma scalar (what `evals.csv` reports) at each ncells. The floor's strong ncells dependence is the **2/n** diagonal effect; the model's **gap above floor** is the sample-size-robust part that actually reflects prediction quality.

In [6]:
import pandas as pd

NCELLS = [30, 50, 80]
rows = []
for name, (t, i) in clouds.items():
    for nc in NCELLS:
        m = gamma_curve(t, i, GAMMAS, nc, N_REPS, kind="model")["mmd"].mean()
        f = gamma_curve(t, None, GAMMAS, nc, N_REPS, kind="floor")["mmd"].mean()
        rows.append({
            "model": name, "ncells": nc,
            "model_mmd": m, "mmd_floor": f,
            "gap_above_floor": m - f, "ratio": m / f,
            "2/n (theory floor)": 2.0 / nc,
        })
table = pd.DataFrame(rows).round(4)
table.to_csv(OUT_DIR / "mmd_floor_vs_ncells.csv", index=False)
print("Note: model_mmd ties back to evals.csv; the floor tracks 2/n as predicted.")
table

Note: model_mmd ties back to evals.csv; the floor tracks 2/n as predicted.


,model,ncells,model_mmd,mmd_floor,gap_above_floor,ratio,2/n (theory floor)
0,M2 IMPACT,30,0.1449,0.0637,0.0812,2.2740,0.0667
1,M2 IMPACT,50,0.1220,0.0381,0.0839,3.1999,0.0400
2,M2 IMPACT,80,0.1063,0.0240,0.0823,4.4271,0.0250
3,M2 scGen,30,0.1822,0.0637,0.1185,2.8593,0.0667
4,M2 scGen,50,0.1581,0.0381,0.1200,4.1475,0.0400
5,M2 scGen,80,0.1439,0.0240,0.1199,5.9940,0.0250
6,M1 scGen,30,0.2012,0.0635,0.1378,3.1704,0.0667
7,M1 scGen,50,0.1829,0.0382,0.1447,4.7905,0.0400
8,M1 scGen,80,0.1690,0.0238,0.1452,7.0946,0.0250
9,M1 IMPACT,30,0.1718,0.0635,0.1083,2.7060,0.0667


## 5. Atlas-reference UMAP with scGen / IMPACT predictions projected

Ported from `presentation_preparation.ipynb` (the recipe behind `umap_atlas_ref_m2_ood_pearson_scgen_impact.pdf`), generalized to M1. Recipe:

1. **Reference** = the `split == "train"` cells of the Pearson HVG object (mouse + human atlas), with the *exact* `toggle_ood` split (`random_state=0`, `test_size=0.2`) so the OOD cells line up with the trained model's `imputed.h5ad`.
2. **Fit** PCA → neighbors → UMAP **only on that reference** (`min_dist=0.3`, `random_state=42`).
3. **Project** the OOD-mouse holdout, the scGen / IMPACT_CellOT predicted-human (`evals_ood_data_space/imputed.h5ad`), and the actual human OOD onto the reference UMAP by kNN in PCA space.

It plots whichever model predictions are available, so it produces a useful figure now (atlas + M1 scGen + actual human) and the full overlay once **M1 IMPACT** finishes training + eval — just re-run §5. Output PNG: `umap_atlas_ref_<group>_ood_pearson_scgen_impact.png`.

In [7]:
import pandas as pd
import anndata as ad
import scanpy as sc
from sklearn.neighbors import NearestNeighbors
from sklearn.model_selection import train_test_split

RESULTS = CELL_GPU / "results"
DATASETS = CELL_GPU / "datasets/speciesot-human-mouse-hvg"

# One entry per experiment line. group/mode map to the on-disk dataset + results dirs.
UMAP_EXPERIMENTS = {
    "M1": dict(group="m1", mode="ood", holdout="CL:0000875",
               title="M1 monocyte (non-classical)", hold_lbl="M1 holdout — mouse"),
    "M2": dict(group="m2", mode="ood", holdout=("CL:0000875", "CL:0000576"),
               title="M2 monocyte", hold_lbl="M2 holdout — mouse"),
}


def _x_dense(X):
    return X.toarray() if hasattr(X, "toarray") else np.asarray(X)


def add_transport(adata, source="mouse", target="human", condition_col="condition"):
    m = {source: "source", target: "target"}
    out = adata.copy()
    out.obs = out.obs.copy()
    out.obs["transport"] = out.obs[condition_col].map(m)
    return out[out.obs["transport"].notna()].copy()


def split_toggle_ood(adata, groupby, holdout, key, mode, random_state=0, test_size=0.2):
    """Reproduce cellot's toggle_ood split so OOD cells align with imputed.h5ad."""
    split = pd.Series(index=adata.obs_names, dtype=object)
    for _, idx in adata.obs.groupby(groupby, observed=False).groups.items():
        tr, te = train_test_split(idx, random_state=random_state, test_size=test_size)
        split.loc[tr] = "train"
        split.loc[te] = "test"
    hv = [holdout] if isinstance(holdout, str) else list(holdout)
    ood_ix = adata.obs_names[adata.obs[key].isin(hv)]
    a, b = train_test_split(ood_ix, random_state=random_state, test_size=0.5)
    if mode == "ood":
        split.loc[a] = "ignore"
        split.loc[b] = "ood"
    else:
        split.loc[a] = "train"
        split.loc[b] = "ood"
    adata.obs["split"] = split.astype("category")
    return adata


def project_onto_ref_umap(pred_X, ref_adata, n_neighbors=10):
    """kNN-project pred cells onto a fitted reference UMAP, in the reference PCA space."""
    pcs = ref_adata.varm["PCs"]
    ref_pca = ref_adata.obsm["X_pca"]
    ref_umap = ref_adata.obsm["X_umap"]
    ref_mean = np.asarray(ref_adata.X.mean(axis=0)).ravel()
    pred_X = np.asarray(pred_X, dtype=np.float32)
    pred_pca = (pred_X - ref_mean) @ pcs
    k = min(n_neighbors, max(1, ref_pca.shape[0] - 1))
    nn = NearestNeighbors(n_neighbors=k, metric="euclidean")
    nn.fit(ref_pca)
    dists, idxs = nn.kneighbors(pred_pca)
    w = 1.0 / (dists + 1e-8)
    w = w / w.sum(axis=1, keepdims=True)
    return np.array([(w[i, :, None] * ref_umap[idxs[i]]).sum(axis=0) for i in range(len(pred_pca))])


def atlas_umap(exp_name):
    e = UMAP_EXPERIMENTS[exp_name]
    rd = RESULTS / f"hvg_pearson_residuals_{e['group']}_{e['mode']}"
    h5 = DATASETS / f"hvg_pearson_residuals_{e['group']}_v07.h5ad"
    if not h5.exists():
        print(f"[{exp_name}] missing dataset {h5}"); return

    d = split_toggle_ood(
        add_transport(ad.read_h5ad(h5)), groupby="condition", holdout=e["holdout"],
        key="cell_type_ontology_term_id", mode=e["mode"], random_state=0, test_size=0.2,
    )
    ct = d.obs["cell_type_ontology_term_id"].astype(str)
    hv_ids = set([e["holdout"]] if isinstance(e["holdout"], str) else list(e["holdout"]))

    # Reference UMAP fit on training-atlas cells only.
    ref = d[d.obs["split"] == "train"].copy()
    ref.X = _x_dense(ref.X).astype(np.float32)
    tr = ref.obs["transport"].astype(str)
    ref.obs["atlas_species"] = np.where(tr == "source", "Atlas — mouse (train)", "Atlas — human (train)")
    n_pcs = int(min(50, ref.n_vars - 1, max(2, ref.n_obs - 1)))
    sc.pp.pca(ref, n_comps=n_pcs)
    sc.pp.neighbors(ref, n_neighbors=min(15, max(2, ref.n_obs - 1)), n_pcs=n_pcs)
    sc.tl.umap(ref, min_dist=0.3, random_state=42)

    ood_m = (d.obs["split"] == "ood") & ct.isin(hv_ids) & (d.obs["transport"] == "source")
    ood_h = (d.obs["split"] == "ood") & ct.isin(hv_ids) & (d.obs["transport"] == "target")
    src_ix = d.obs_names[ood_m]
    u_m = project_onto_ref_umap(_x_dense(d[ood_m].X).astype(np.float32), ref)
    u_h = project_onto_ref_umap(_x_dense(d[ood_h].X).astype(np.float32), ref)

    # Project whichever model predictions are available.
    overlays = []
    for model_dir, label, color in [
        ("scgen", "scGen predicted human", "#800080"),
        ("impact_cellot", "IMPACT_CellOT predicted human", "#00BFBF"),
    ]:
        p = rd / model_dir / "evals_ood_data_space" / "imputed.h5ad"
        if not p.exists():
            print(f"[{exp_name}] {label}: imputed.h5ad not ready yet — skipping overlay")
            continue
        a = ad.read_h5ad(p)
        if not a.obs_names.equals(src_ix):
            print(f"[{exp_name}] {label}: obs_names != OOD source — skipping")
            continue
        X = _x_dense(a.X).astype(np.float32)
        if X.shape[1] != ref.n_vars:
            print(f"[{exp_name}] {label}: gene-dim mismatch — skipping"); continue
        overlays.append((label, color, project_onto_ref_umap(X, ref)))

    fig, ax = plt.subplots(figsize=(9, 7))
    ur = ref.obsm["X_umap"]
    for lab, color in [("Atlas — mouse (train)", "#aec7e8"), ("Atlas — human (train)", "#98df8a")]:
        m = ref.obs["atlas_species"].astype(str) == lab
        ax.scatter(ur[m, 0], ur[m, 1], s=7, c=color, alpha=0.32, edgecolors="none",
                   label=f"{lab} (n={int(m.sum())})", zorder=1)
    ax.scatter(u_m[:, 0], u_m[:, 1], s=24, c="#6baed6", alpha=0.9, edgecolors="none",
               label=f"{e['hold_lbl']} (n={len(u_m)})", zorder=4)
    z = 5
    for label, color, U in overlays:
        ax.scatter(U[:, 0], U[:, 1], s=24, c=color, alpha=0.88, edgecolors="none",
                   label=f"{label} (n={len(U)})", zorder=z); z += 1
    ax.scatter(u_h[:, 0], u_h[:, 1], s=24, c="#d62728", alpha=0.92, edgecolors="none",
               label=f"Actual human OOD (n={len(u_h)})", zorder=z)
    ax.set_xlabel("UMAP1"); ax.set_ylabel("UMAP2")
    ax.set_title(f"Training atlas UMAP (Pearson HVG, {e['title']}) + model overlays", fontsize=10, fontweight="bold")
    ax.legend(loc="best", fontsize=7, framealpha=0.94)
    plt.tight_layout()
    stem = OUT_DIR / f"umap_atlas_ref_{e['group']}_{e['mode']}_pearson_scgen_impact"
    fig.savefig(stem.with_suffix(".png"), dpi=300, bbox_inches="tight")
    print(f"[{exp_name}] saved {stem.with_suffix('.png')}")
    plt.show()

In [8]:
# UMAP POSTPONED (2026-06-02): the reference-UMAP fit is slow on the login node.
# The recipe above is ready — uncomment to run interactively (ideally on a compute
# node / in a fresh kernel), once you want the atlas-projection figures.
# atlas_umap("M1")
# atlas_umap("M2")
print("UMAP step is defined but not executed (postponed). Uncomment atlas_umap(...) to run.")

UMAP step is defined but not executed (postponed). Uncomment atlas_umap(...) to run.


## 6. R² / MMD comparison — M1 vs M2, OOD vs IID

Headline metrics pulled straight from each model's `evals.csv` at `ncells=80, nfeatures=all`:
- **R²** = `(r2-means)²` (the upstream stores Pearson r under the `r2-means` label; we square it, per `conceptual_framework.md` §5.5).
- **MMD** = mean of the `mmd` rows; **`mmd_floor`** is the split-half self-MMD (§4, the best achievable), and **`mmd_ceiling`** = `MMD(mouse control, real human)` is the no-transport / identity-baseline gap (the worst a sensible model should incur).
- **`gap_above_floor` = MMD − floor** (sample-size-robust error) and **`frac_gap_closed` = (ceiling − MMD) / (ceiling − floor)** (1.0 = reached the floor, 0 = no better than identity) place the model on the floor↔ceiling axis.
- **`mean_JS`** = mean per-gene Jensen-Shannon divergence between treated and imputed marginals (`cellot.losses.compute_marginal_divergence`) — the interpretable per-gene complement to the joint MMD.

> **Important finding (and why M1's MMD looked confusing).** Here the **ceiling comes out *below* the model MMD**, so `frac_gap_closed` is *negative*. The ceiling = `MMD(mouse control, real human)` is only ~0.09, but the model's `MMD(imputed, human)` is ~0.11-0.14. Because the OOD mouse and human cells are *matched* lung monocytes, they are already distributionally close (small cross-species gap), yet the OT transport pushes cells far from mouse (`MMD(control, imputed)` ~0.23) and **overshoots**, landing farther from real human than the untransported mouse was. So on the full-distribution (MMD) metric the model does *worse than doing nothing*, even while nailing the per-gene mean (R^2 ~0.94). A negative `frac_gap_closed` is meaningful, not a bug: it flags a regime where there is little distributional gap to close and OT mainly adds spread. This is the crux of why M2 -> M1 raised R^2 but did not tighten MMD.

"IID" vs "OOD" is the **training mode** of the experiment (`hvg_pearson_residuals_{group}_{iid|ood}`): in IID the model saw the holdout cell type during training; in OOD it never did. Availability:

- **M2**: both IID and OOD trained + evaluated.
- **M1**: **OOD only** — there is no `hvg_pearson_residuals_m1_iid` experiment yet (would need a separate generate + train). Its row is shown as `n/a`.

In [9]:
import sys as _sys
if str(CELL_GPU) not in _sys.path:
    _sys.path.insert(0, str(CELL_GPU))
from cellot.losses import compute_marginal_divergence  # validate the library KL/JS

NCELLS_TBL = 80
LINES = {"M1": "m1", "M2": "m2"}
MODES = ["ood", "iid"]
MODELS = [("impact_cellot", "IMPACT"), ("scgen", "scGen")]


def _eval_csv_metrics(csv_path, ncells=NCELLS_TBL):
    df = pd.read_csv(csv_path)
    sub = df[(df["nfeatures"] == "all") & (df["ncells"] == ncells)]
    r2 = (sub.loc[sub["metric"] == "r2-means", "value"].astype(float) ** 2).mean()
    mmd = sub.loc[sub["metric"] == "mmd", "value"].astype(float).mean()
    return r2, mmd


def _npz_for(group, mode, model_dir):
    p = RESULTS / f"hvg_pearson_residuals_{group}_{mode}" / model_dir / "evals_ood_data_space" / "eval_clouds.npz"
    return np.load(p, allow_pickle=False) if p.exists() else None


rows = []
for line, group in LINES.items():
    for mode in MODES:
        base = RESULTS / f"hvg_pearson_residuals_{group}_{mode}"
        # treated + control are shared across the two models of an experiment
        treated = control = None
        for md in ("impact_cellot", "scgen"):
            z = _npz_for(group, mode, md)
            if z is not None:
                treated = z["treated"]
                control = z["control"] if "control" in z.files else None
                break
        floor = (gamma_curve(treated, None, GAMMAS, NCELLS_TBL, N_REPS, kind="floor")["mmd"].mean()
                 if treated is not None else np.nan)
        # ceiling = MMD(control mouse, treated human): the no-transport / identity gap
        ceiling = (gamma_curve(control, treated, GAMMAS, NCELLS_TBL, N_REPS, kind="model")["mmd"].mean()
                   if (treated is not None and control is not None) else np.nan)
        for model_dir, mlabel in MODELS:
            csv = base / model_dir / "evals_ood_data_space" / "evals.csv"
            if not csv.exists():
                rows.append({"line": line, "setting": mode, "model": mlabel, "R2": np.nan,
                             "MMD": np.nan, "mmd_floor": np.nan, "mmd_ceiling": np.nan,
                             "gap_above_floor": np.nan, "frac_gap_closed": np.nan,
                             "mean_JS": np.nan, "status": "n/a — not trained"})
                continue
            r2, mmd = _eval_csv_metrics(csv)
            z = _npz_for(group, mode, model_dir)
            mean_js = (compute_marginal_divergence(z["treated"], z["imputed"])["mean_js"]
                       if z is not None else np.nan)
            frac = ((ceiling - mmd) / (ceiling - floor)
                    if (ceiling is not None and not np.isnan(ceiling) and ceiling > floor) else np.nan)
            rows.append({"line": line, "setting": mode, "model": mlabel, "R2": r2, "MMD": mmd,
                         "mmd_floor": floor, "mmd_ceiling": ceiling,
                         "gap_above_floor": (mmd - floor) if treated is not None else np.nan,
                         "frac_gap_closed": frac, "mean_JS": mean_js, "status": "ok"})

cmp_table = pd.DataFrame(rows)
_numcols = ["R2", "MMD", "mmd_floor", "mmd_ceiling", "gap_above_floor", "frac_gap_closed", "mean_JS"]
cmp_table[_numcols] = cmp_table[_numcols].round(4)
cmp_table.to_csv(OUT_DIR / "r2_mmd_comparison_m1_m2.csv", index=False)
cmp_table

,line,setting,model,R2,MMD,mmd_floor,mmd_ceiling,gap_above_floor,frac_gap_closed,mean_JS,status
0,M1,ood,IMPACT,0.9407,0.1375,0.0238,0.0896,0.1137,-0.7277,0.4653,ok
1,M1,ood,scGen,0.9094,0.1677,0.0238,0.0896,0.1439,-1.1865,0.4901,ok
2,M1,iid,IMPACT,NaN,NaN,NaN,NaN,NaN,NaN,NaN,n/a — not trained
3,M1,iid,scGen,NaN,NaN,NaN,NaN,NaN,NaN,NaN,n/a — not trained
4,M2,ood,IMPACT,0.9301,0.1080,0.0240,0.0862,0.0840,-0.3507,0.4507,ok
5,M2,ood,scGen,0.8916,0.1458,0.0240,0.0862,0.1218,-0.9595,0.4736,ok
6,M2,iid,IMPACT,0.9674,0.1207,0.0240,0.0862,0.0967,-0.5551,0.4516,ok
7,M2,iid,scGen,0.9364,0.1417,0.0240,0.0862,0.1177,-0.8927,0.4664,ok


## 7. Figure-G-style per-gene marginals (KL/JS view)

The paper's Figure G shows, for a few marker genes in the OOD setting, the marginal expression density of **Treated** vs **CellOT** vs **scGen** — a good model's curve overlays the treated curve. Here we reproduce that for M1, picking the **6 genes where IMPACT's marginal diverges most from treated** (largest per-gene JS), and annotate each panel with the per-gene JS for IMPACT and scGen. This is the interpretable, per-gene complement to the joint MMD: JS quantifies, gene by gene, how far each model's marginal is from the real one.

In [10]:
from scipy.stats import gaussian_kde

GROUP, MODE = "m1", "ood"
z_imp = _npz_for(GROUP, MODE, "impact_cellot")
z_sg = _npz_for(GROUP, MODE, "scgen")
treated = z_imp["treated"]
imp = z_imp["imputed"]
sg = z_sg["imputed"]
genes = [str(g) for g in z_imp["genes"]] if "genes" in z_imp.files else [str(i) for i in range(treated.shape[1])]

js_imp = compute_marginal_divergence(treated, imp)["js"]
js_sg = compute_marginal_divergence(treated, sg)["js"]
top = np.argsort(js_imp)[::-1][:6]   # 6 genes where IMPACT's marginal is most off


def _density(ax, arr, color, label, xs):
    try:
        ax.plot(xs, gaussian_kde(arr)(xs), color=color, lw=1.8, label=label)
    except Exception:
        ax.hist(arr, bins=30, density=True, histtype="step", color=color, label=label)


fig, axes = plt.subplots(2, 3, figsize=(16, 9))
for ax, gi in zip(axes.ravel(), top):
    lo = min(treated[:, gi].min(), imp[:, gi].min(), sg[:, gi].min())
    hi = max(treated[:, gi].max(), imp[:, gi].max(), sg[:, gi].max())
    xs = np.linspace(lo, hi, 200) if hi > lo else np.linspace(lo - 1, lo + 1, 200)
    _density(ax, treated[:, gi], "#1f4e96", "Treated", xs)
    _density(ax, imp[:, gi], "#f05a5f", "IMPACT_CellOT", xs)
    _density(ax, sg[:, gi], "#b0b0b0", "scGen", xs)
    ax.set_title(f"{genes[gi]}\nJS  IMPACT={js_imp[gi]:.3f}  scGen={js_sg[gi]:.3f}", fontsize=9)
    ax.set_xlabel("expression (log-norm)")
    ax.set_ylabel("o.o.d. density")
axes.ravel()[0].legend(fontsize=8)
fig.suptitle("M1 OOD per-gene marginals — 6 most IMPACT-divergent genes (Figure-G style)",
             fontweight="bold")
fig.tight_layout()
fig.savefig(OUT_DIR / "figureG_m1_marginals.png", dpi=150, bbox_inches="tight")
print("saved", OUT_DIR / "figureG_m1_marginals.png")
print(f"mean JS over all genes: IMPACT={js_imp.mean():.4f}  scGen={js_sg.mean():.4f}")
plt.show()

saved /n/holylabs/mooney_lab/Lab/junyizhou/speciesOT/speciesOT/baseline/analysis/m1_pearson_outputs/figureG_m1_marginals.png
mean JS over all genes: IMPACT=0.4653  scGen=0.4901
